In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned")

In [3]:
df = pd.DataFrame(dataset)

df.head()

,train
0,"{'source': 'evol_instruct', 'prompt': 'Can you..."
1,"{'source': 'evol_instruct', 'prompt': 'Suppose..."
2,"{'source': 'evol_instruct', 'prompt': 'Identif..."
3,"{'source': 'evol_instruct', 'prompt': 'How can..."
4,"{'source': 'evol_instruct', 'prompt': 'Can you..."


In [4]:
import json
# select the first row (change index if you want a different row)
row = df.iloc[0]
print(json.dumps(row.to_dict(), indent=2, ensure_ascii=False))

{
  "train": {
    "source": "evol_instruct",
    "prompt": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}",
    "chosen": [
      {
        "content": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++

In [5]:
import jsonlines
from pathlib import Path

In [6]:
# Write to JSON lines file
output_folder = Path(Path.cwd() / "content")
output_folder.mkdir(exist_ok=True)

# Use JSON Lines (.jsonl) so each item is a separate JSON object on its own line
file_path = output_folder / 'QA_prompts_and_completions.jsonl'
# Write using the jsonlines library for correct JSONL formatting
items = []
for idx, row in df.iterrows():
    row_dic = row.iloc[0]
    row_dic["index"] = idx
    items.append(row_dic)

with jsonlines.open(str(file_path), mode='w') as writer:
    writer.write_all(items)

## Download model

In [7]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


model_name = "TinyLlama/TinyLlama_v1.1"

model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto") #lower precision to finetune
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.gradient_checkpointing_enable()
model.config.use_cache = False


Loading weights:   2%|▏         | 4/201 [00:04<04:07,  1.26s/it]Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "C:\Users\Utente\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Utente\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\transformers\safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\transformers\safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\transformers\safetensors_convers

In [9]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm(

In [10]:
#QLoRA for faster finetuning
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, #rank of matrices
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'], #other layers are frozen
    lora_dropout=0.05,
    bias='none',
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [11]:
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [12]:
ref_model = model #frozen model
ref_model.eval()

train_model = model
train_model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [13]:
print(dataset['train'][0]['prompt'])
print(dataset['train'][0]['chosen'][1]['content']) #chosen response


Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:
#include <iostream>
#include <string>
using namespace std;
int main() {
    string country;
    // prompt user for input
    cout << "Enter the name of a country: ";
    cin >> country;
    // check if country borders the Mediterranean Sea
    // [C++ code]
    return 0;
}
Here's a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:

#include <iostream>
#include <string>
#include <set>
#include <map>
#include <algorithm>

using namespace std;

int main() {
    // store countries and their bordering seas in a map
    map<string, set<string>> countries;
    countries["Algeria"] = {"Mediterranean Sea", "North African Coast"};
    countries["France"] = {"Mediterranean Sea", "English Channel"};
    countries["Italy"] = {"Mediterranean Sea", "Adriatic Sea"};
    c

In [14]:
print(dataset['train'][0]['rejected'][1]['content']) #rejected response

Sure, here is the program using the C++11 algorithm "cds::algorithm::GreaterEqual":
#include <iostream>
#include <string>
#include <algorithm>
#include <vector>
#include <cctype>

using namespace std;

int main() {
    string country;
    cout << "Enter the name of a country: ";
    cin >> country;
    std::vector<string> vec;
    vec.push_back(country);
    size_t index = std::find_if(vec.begin(), vec.end(), [](const string& s) {
        return std::any_of(s.begin(), s.end(), [](const char& c) {
            return c == '}}';
    });
    if (index != vec.end()) {
        if ((*index)[0] == 'M') {
            cout << "Country is bordered by the Mediterranean Sea." << endl;
        } else {
            cout << "Country does not border the Mediterranean Sea." << endl;
        }
    } else {
        cout << "Country is not found." << endl;
    }
    return 0;
}


In [15]:
def _util_fn(batch, tokenizer, max_length = 1024):
    prompts = [x['prompt'] for x in batch]
    chosen = [x['chosen'][1]['content'] for x in batch]
    rejected = [x['rejected'][1]['content'] for x in batch]

    chosen_texts = [p + c for p, c in zip(prompts, chosen)]
    rejected_texts = [p + c for p, c in zip(prompts, rejected)]

    chosen_enc = tokenizer(
        chosen_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    rejected_enc = tokenizer(
        rejected_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    prompt_lens = [ #useful for masking
        len(tokenizer(p)) for p in prompts
    ]

    return {
        "chosen_input_ids": chosen_enc["input_ids"],
        "chosen_attention_mask": chosen_enc["attention_mask"],
        "rejected_input_ids": rejected_enc["input_ids"],
        "rejected_attention_mask": rejected_enc["attention_mask"],
        "prompt_lens": torch.tensor(prompt_lens)
    }

In [16]:
def compute_logps(model, input_ids, attention_mask, prompt_lens):
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    log_probs = F.log_softmax(logits, dim=-1)

    shift_logits = log_probs[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    selected = shift_logits.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1) #selected[i, t] = logprob(t+1 | tokens<=t)

    mask = torch.ones_like(shift_labels) #general mask

    for idx, pl in enumerate(prompt_lens):
        mask[idx, :pl-1] = 0
    
    selected = selected * mask #maintain only logprobs of otkens not in prompt

    #for each sentence, return the total logprob (summing in last dimension)
    return selected.sum(dim=-1)

In [24]:
def dpo_loss(train_model, ref_model, batch, beta=0.1):

    train_chosen_logps = compute_logps(
        train_model,
        batch['chosen_input_ids'],
        batch['chosen_attention_mask'],
        batch['prompt_lens']
    )
    
    train_rejected_logps = compute_logps(
        train_model,
        batch['rejected_input_ids'],
        batch['rejected_attention_mask'],
        batch['prompt_lens']
    )

    with torch.no_grad():
        ref_chosen_logps = compute_logps(
            ref_model,
            batch['chosen_input_ids'],
            batch['chosen_attention_mask'],
            batch['prompt_lens']
        )
        
        ref_rejected_logps = compute_logps(
            ref_model,
            batch['rejected_input_ids'],
            batch['rejected_attention_mask'],
            batch['prompt_lens']
        )
    
    diff = (train_chosen_logps - ref_chosen_logps) - (train_rejected_logps - ref_rejected_logps)

    return - F.logsigmoid(beta * diff).mean()

In [18]:
checkpoint_path  =  Path(Path.cwd() / 'checkpoints')
checkpoint_path.mkdir(exist_ok=True)


In [35]:
def train(train_model, ref_model, optimizer, scaler, epochs, dataloader, tokenizer):
    for epoch in range(epochs):
        print(f"STARTING EPOCH {epoch+1}...")
        for batch in dataloader:

            batch = {k: v.to(device) for k,v in batch.items()}

            with torch.autocast(device_type=device):
                loss = dpo_loss(train_model, ref_model, batch)
            
            scaler.scale(loss).backward() #avoid underflow (F16 generates tiny gradients)

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        if epoch % 5 == 0 :
            model.save_pretrained(checkpoint_path / f"epoch_{epoch+1}")
            tokenizer.save_pretrained(checkpoint_path / f"epoch_{epoch+1}")

        print(f"LOSS: {loss.item()}")

In [36]:
def collate_fn(batch):
    return _util_fn(batch, tokenizer)

In [37]:
from torch import optim
optimizer = optim.AdamW(
    filter(lambda x: x.requires_grad, train_model.parameters()),
    lr = 5e-6
)

scaler = torch.GradScaler()
dataloader = DataLoader(dataset['train'], batch_size=2, shuffle=True, collate_fn=lambda x: collate_fn(x), num_workers=0)

In [38]:
tokenizer.pad_token = tokenizer.eos_token  
train(train_model, ref_model, optimizer, scaler, 50, dataloader, tokenizer)

STARTING EPOCH 1...


KeyboardInterrupt: 